In [1]:
%pip install optuna
!git clone https://github.com/utkuayten/CS401-soffritto

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 29.9 MB/s eta 0:00:00
Cloning into 'CS401-soffritto'...
remote: Enumerating objects: 1690, done.
remote: Counting objects: 100% (166/166), done.
remote: Compressing objects: 100% (109/109), done.
remote: Total 1690 (delta 101), reused 109 (delta 57), pack-reused 1524 (from 2)
Receiving objects: 100% (1690/1690), 764.79 MiB | 17.75 MiB/s, done.
Resolving deltas: 100% (924/924), done.
Updating files: 100% (522/522), done.


In [2]:
%cd CS401-soffritto

/content/CS401-soffritto


In [3]:
import pandas as pd 

df = pd.read_csv('optuna_BOCO_H1.csv')
df.sort_values(by='value', inplace=True)
best_params = df.iloc[0]
print("Best Parameters :",)
hp = {
    k.replace("params_", ""): best_params[k]
    for k in best_params.index
    if k.startswith("params_")
}
print(hp)

Best Parameters :
{'d_ff': np.int64(2048), 'd_layers': np.int64(3), 'd_model': np.int64(1024), 'dropout': np.float64(0.0740311182114872), 'e_layers': np.int64(4), 'factor': np.int64(5), 'learning_rate': np.float64(3.961216590164204e-05), 'n_heads': np.int64(4), 'weight_decay': np.float64(3.420327610666098e-05)}


In [4]:
import os
import sys
from argparse import Namespace

cwd = os.getcwd()

if os.path.isdir(os.path.join(cwd, 'transofritto')):
    PROJECT_ROOT = cwd
elif os.path.isdir(os.path.join(cwd, 'CS401-soffritto')):
    PROJECT_ROOT = os.path.join(cwd, 'CS401-soffritto')
else:
    PROJECT_ROOT = cwd  # fallback

print("PROJECT_ROOT =", PROJECT_ROOT)

TRANSOFTRITTO_DIR = (
    PROJECT_ROOT if os.path.basename(PROJECT_ROOT) == "transofritto"
    else os.path.join(PROJECT_ROOT, "transofritto")
)

print("TRANSOFTRITTO_DIR =", TRANSOFTRITTO_DIR)

if TRANSOFTRITTO_DIR not in sys.path:
    sys.path.insert(0, TRANSOFTRITTO_DIR)

from train_intra_cell import main as train_intra_cell_main

PROJECT_ROOT = /content/CS401-soffritto
TRANSOFTRITTO_DIR = /content/CS401-soffritto/transofritto


In [20]:
import numpy as np
from argparse import Namespace

# hp: best_params'tan gelen dict'in
# ÖNCE tüm np.int64 / np.float64 değerleri normal Python int/float'a çeviriyoruz
hp_clean = {
    k: (v.item() if isinstance(v, np.generic) else v)
    for k, v in hp.items()
}

cell = "H1"  

chroms_all = list(range(1, 23))

val_chroms   = [6]
test_chroms  = [9]
train_chroms = [c for c in chroms_all if c not in val_chroms + test_chroms]

setting_name = f"{cell}_FINAL_best_optuna"

args = Namespace(
    # --- data / CV config ---
    cell=cell,
    train_chroms=train_chroms,
    val_chroms=val_chroms,
    test_chroms=test_chroms,
    setting=setting_name,
    checkpoints=f"checkpoints/{setting_name}",

    # --- BEST HYPERPARAMS FROM *.csv  ---
    **hp_clean,   # <--- SADECE BURASI DEĞİŞTİ

    # --- fixed stuff (same as in your Optuna objective) ---
    seq_len=32,
    batch_size=256,
    
    label_len=16,   # seq_len // 2
    pred_len=1,

    enc_in=9,
    dec_in=1,

    c_out=16,
    activation='gelu',
    attn='prob',

    train_epochs=15,   # you can increase for final training
    patience=5,
    lradj='type1',
    num_workers=8,
    use_multi_gpu=False,
    gpu=0,
    devices='0',
    selected_cols=[
        'H3K27ac', 'H3K27me3', 'H3K36me3', 'H3K4me1',
        'H3K4me3', 'H3K9me3', 'GC_content', 'gene_density', '2-stage'
    ],
)

result = train_intra_cell_main(args)
print("Final training result:", result)

Namespace(cell='H1', train_chroms=[1, 2, 3, 4, 5, 7, 8, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22], val_chroms=[6], test_chroms=[9], setting='H1_FINAL_best_optuna', checkpoints='/content/CS401-soffritto/transofritto/checkpoints', d_ff=2048, d_layers=3, d_model=1024, dropout=0.0740311182114872, e_layers=4, factor=5, learning_rate=3.961216590164204e-05, n_heads=4, weight_decay=3.420327610666098e-05, seq_len=32, batch_size=256, label_len=16, pred_len=1, enc_in=9, dec_in=1, c_out=16, activation='gelu', attn='prob', train_epochs=15, patience=5, lradj='type1', num_workers=8, use_multi_gpu=False, gpu=0, devices='0', selected_cols=['H3K27ac', 'H3K27me3', 'H3K36me3', 'H3K4me1', 'H3K4me3', 'H3K9me3', 'GC_content', 'gene_density', '2-stage'], root_path='/content/CS401-soffritto/transofritto/data', data_path='/content/CS401-soffritto/transofritto/data/H1_genomic.csv', results_path='/content/CS401-soffritto/transofritto/results', model='informer', target='target_1', freq='w', embed='timeF'

KeyboardInterrupt: 

In [19]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

true_path = "soffritto/predictions/H1_chr9_pred_intra_cell_line.npy_true.npy"
pred_path = "soffritto/predictions/H1_chr9_pred_intra_cell_line.npy"

# Load as numpy
p_np = np.load(true_path)
q_np = np.load(pred_path)

# To torch tensors (float)
p = torch.from_numpy(p_np).float()  # target
q = torch.from_numpy(q_np).float()  # prediction

log_q = torch.log(q)   # ⚠ if q has zeros, this will give -inf (mathematically infinite KL)

criterion = nn.KLDivLoss(reduction="batchmean")
kl = criterion(log_q, p)

print("KL(true || pred) =", kl.item())

KL(true || pred) = 0.03883087635040283
